<h2 align="center">Machine Learning Model Development</h2>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor,
    VotingRegressor
)
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [3]:
df = pd.read_excel('../../data/processed/3rd_one_year_mine_data_cleaned.xlsx')
df.columns = df.columns.str.strip()
df.head()

,Date,PM10,PM2.5,Temperature,Relative_humidity,RF-CUMULATIVE,Wind_speed,Wind_direction,Road_width,Distance,Production_ROM,Production_OB,Production_ROM+OB,Vehicle Capacity,LOCATION,SEASON
0,2025-03-04,70.8,24.1,36.1,20.4,0.0,8.0,NorthWest,7,30,30000,75000,105000,2.5,Near Office,Summer
1,2025-03-08,69.5,32.6,36.0,22.4,0.0,8.0,NorthWest,7,30,30000,75000,105000,2.5,Near Office,Summer
2,2025-03-11,65.2,30.4,35.5,21.8,0.0,8.0,NorthWest,7,30,30000,75000,105000,2.5,Near Office,Summer
3,2025-03-15,76.8,26.5,34.8,38.5,0.0,6.0,NorthWest,7,30,30000,75000,105000,2.5,Near Office,Summer
4,2025-03-18,58.4,20.4,35.1,31.4,0.0,7.0,NorthWest,7,30,30000,75000,105000,2.5,Near Office,Summer


In [4]:
df = pd.get_dummies(df, columns=['Wind_direction'])

In [5]:
df.columns

Index(['Date', 'PM10', 'PM2.5', 'Temperature', 'Relative_humidity',
       'RF-CUMULATIVE', 'Wind_speed', 'Road_width', 'Distance',
       'Production_ROM', 'Production_OB', 'Production_ROM+OB',
       'Vehicle Capacity', 'LOCATION', 'SEASON', 'Wind_direction_NorthWest'],
      dtype='object')

In [7]:
features = [
    "Temperature",
    "Relative_humidity",
    "RF-CUMULATIVE",
    "Wind_speed",
    'Wind_direction_NorthWest',
    "Distance",
    'Production_ROM',
    'Production_OB',
    'Production_ROM+OB',
    "Vehicle Capacity",
]

target_pm10 = "PM10"
target_pm25 = "PM2.5"

In [8]:
X = df[features]
y = df[target_pm10]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

<h2 align="center">Linear Regression Model</h2>

In [9]:
from sklearn.impute import SimpleImputer

# Impute missing feature values (use median to be robust to outliers)
imputer = SimpleImputer(strategy="median")
imputer.fit(X_train)

X_train_imputed = pd.DataFrame(
	imputer.transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_imputed = pd.DataFrame(
	imputer.transform(X_test), columns=X_test.columns, index=X_test.index
)

lr = LinearRegression()
lr.fit(X_train_imputed, y_train)
pred = lr.predict(X_test_imputed)

In [10]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(mae)
print(rmse)
print(r2)

6.84195983043346
8.570568819636527
0.19965596018821352


<h2 align="center">Decision Tree Model</h2>

In [11]:
# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [12]:
mae = mean_absolute_error(y_test, dt_pred)
rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
r2 = r2_score(y_test, dt_pred)

print(mae)
print(rmse)
print(r2)

9.766379310344828
12.502727288682753
-0.7032050603796696


<h2 align="center">Random Forest Model</h2>

In [13]:
# Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [14]:
mae = mean_absolute_error(y_test, rf_pred)
rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
r2 = r2_score(y_test, rf_pred)

print(mae)
print(rmse)
print(r2)

7.252620689655173
9.341306525736037
0.04923613269733573


<h2 align="center">Bagging Model</h2>

In [15]:
# Bagging
try:
    bagging = BaggingRegressor(
        estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )
except TypeError:
    bagging = BaggingRegressor(
        base_estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )

bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)

In [16]:
mae = mean_absolute_error(y_test, bagging_pred)
rmse = np.sqrt(mean_squared_error(y_test, bagging_pred))
r2 = r2_score(y_test, bagging_pred)

print(mae)
print(rmse)
print(r2)

7.342836206896553
9.412156882512877
0.03475905354314868


<h2 align="center">SVR Model</h2>

In [17]:
# SVR
svr = SVR(kernel="rbf")
svr.fit(X_train, y_train)
svr_pred = svr.predict(X_test)

In [18]:
mae = mean_absolute_error(y_test, svr_pred)
rmse = np.sqrt(mean_squared_error(y_test, svr_pred))
r2 = r2_score(y_test, svr_pred)

print(mae)
print(rmse)
print(r2)

7.537001172221001
9.398248198379989
0.037609688552571185


<h2 align="center">XGBoost Model</h2>

In [19]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

In [20]:
mae = mean_absolute_error(y_test, xgb_pred)
rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
r2 = r2_score(y_test, xgb_pred)

print(mae)
print(rmse)
print(r2)

7.551080473538102
9.664905955701242
-0.017777127992851582


In [21]:
# Compare model performance in a table

results = []

models = {
    "Linear Regression": pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "Bagging": bagging_pred,
    "SVR": svr_pred,
    "XGBoost": xgb_pred,
}

for model_name, y_pred in models.items():
    results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R²": r2_score(y_test, y_pred)
    })

# Voting Regressor
try:
    voting = VotingRegressor([
        ("lr", LinearRegression()),
        ("dt", DecisionTreeRegressor(random_state=42)),
        ("rf", RandomForestRegressor(n_estimators=200, random_state=42)),
        ("xgb", xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=200,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ))
    ])
    voting.fit(X_train, y_train)
    voting_pred = voting.predict(X_test)

    results.append({
        "Model": "Voting",
        "MAE": mean_absolute_error(y_test, voting_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, voting_pred)),
        "R²": r2_score(y_test, voting_pred)
    })
except Exception as e:
    print("Voting model could not be evaluated:", e)

results_df = pd.DataFrame(results)
results_df

,Model,MAE,RMSE,R²
0,Linear Regression,6.841960,8.570569,0.199656
1,Decision Tree,9.766379,12.502727,-0.703205
2,Random Forest,7.252621,9.341307,0.049236
3,Bagging,7.342836,9.412157,0.034759
4,SVR,7.537001,9.398248,0.037610
5,XGBoost,7.551080,9.664906,-0.017777
6,Voting,7.215801,9.182552,0.081278


In [22]:
Path("../../data/processed").mkdir(parents=True, exist_ok=True)
results_df.to_excel(
    "../../data/processed/results_data_third1.xlsx",
    index=False
)